# AIMessage: BaseMessage

The `AIMessage` class represents a complete message generated by an AI or chat model.

It stores the model’s response content together with standardized tool calls, invalid tool calls, token usage, response metadata, and provider-specific data.

## Import
```python
from langchain_core.messages import AIMessage # Complete response returned by an AI/chat model.
```

## Class
### `AIMessage`
Creates a message representing an AI-generated response.
* **Syntax:**
  ```python
  AIMessage(
      content: str | list[str | dict[Any, Any]] | None = None,
      # Raw response content.
      # str:A normal text response.
      # list[str | dict]: A response containing multiple content items such as text, reasoning, images, audio, files, tool calls, or provider-specific blocks.
      # content is optional only when content_blocks is supplied.

      content_blocks: list[ContentBlock] | None = None,
      # Standard typed LangChain content blocks.
      # Use this instead of content when constructing the message from standardized blocks.

      additional_kwargs: dict[Any, Any] = {},
      # Extra provider-specific response data.
      # It may contain raw information that has not been converted into a standard AIMessage field.

      response_metadata: dict[Any, Any] = {},
      # Metadata describing the model response.
      # It may contain the model name, finish reason, request ID, response headers, log probabilities, provider name, or provider-specific usage details.

      name: str | None = None,
      # Optional human-readable name for the message.
      # Whether this value is used depends on the model provider.

      id: str | None = None,
      # Optional unique identifier for the message.
      # It is normally supplied by the model provider.
      # Numeric IDs are automatically converted to strings.

      tool_calls: list[ToolCall] = [],
      # Valid tool calls requested by the model.
      # Each item contains the tool name, parsed arguments, and an optional call ID.

      invalid_tool_calls: list[InvalidToolCall] = [],
      # Tool calls whose arguments could not be parsed successfully.
      # Each item may contain the tool name, original argument text, call ID, and parsing error.

      usage_metadata: UsageMetadata | None = None,
      # Standardized token-usage information.
      # None means token usage was not supplied by the model provider.

      type: Literal["ai"] = "ai",
      # Identifies the object as an AI message during serialization and deserialization.
      # This value should normally not be changed.

      **kwargs: Any
      # Additional fields accepted by the underlying Pydantic model.
  ) -> None # Returns a new AIMessage object.
  ```

The class supports two constructor forms:
1. Without content_block
  ```python
    AIMessage(
        content: str | list[str | dict[Any, Any]],
        **kwargs: Any,
    ) -> None # Creates the message from raw text or raw content items.
  ```

2. with content_block
  ```python
    AIMessage(
        content: str | list[str | dict[Any, Any]] | None = None,
        content_blocks: list[ContentBlock] | None = None,
        **kwargs: Any,
    ) -> None # Creates the message from raw content or standardized content blocks.
  ```



In [9]:
from langchain_core.messages import AIMessage
from langchain_core.messages.content import create_tool_call

tool_call = create_tool_call( # Creates a tool request generated by the AI.
    name="add", # Specifies the tool to execute.
    args={"a": 10, "b": 20}, # Stores the arguments for the tool.
    id="call_1", # Assigns an identifier to the tool call.
)

message = AIMessage( # Creates a complete AI-generated message.
    content="I will add the two numbers.", # Stores the AI's text response.
    id="message_1", # Stores the message identifier.
    tool_calls=[tool_call], # Stores the tool requested by the AI.
    response_metadata={"model": "test-model", "finish_reason": "tool_calls"}, # Stores response details.
    usage_metadata={ # Stores standardized token usage.
        "input_tokens": 8,
        "output_tokens": 6,
        "total_tokens": 14,
    },
)

def add(a: int, b: int) -> int: # Defines the tool requested by the AI.
    return a + b # Returns the sum.

result = add(**message.tool_calls[0]["args"]) # Executes the requested tool using its arguments.

print("AI:", message.content) # Output: I will add the two numbers.
print("Tool:", message.tool_calls[0]["name"]) # Output: add
print("Result:", result) # Output: 30
print("Tokens:", message.usage_metadata["total_tokens"]) # Output: 14

AI: I will add the two numbers.
Tool: add
Result: 30
Tokens: 14


## Supporting Types
### `ToolCall`

```python
ToolCall = {
    "name": str, # Name of the tool requested by the model.
    "args": dict[str, Any], # Parsed arguments that should be passed to the tool.
    "id": str | None, # Identifier used to match this call with its ToolMessage result.
    "type": "tool_call", # Identifies the dictionary as a valid tool call.
}
```

### `InvalidToolCall`
```python
InvalidToolCall = {
    "name": str | None,# Tool name, when it could be identified.
    "args": str | None, # Original unparsed tool-argument text.
    "id": str | None, # Identifier of the attempted tool call.
    "error": str | None, # Reason the tool call could not be parsed.
    "type": "invalid_tool_call", # Identifies the dictionary as an invalid tool call.
}
```

### `UsageMetadata`
```python
UsageMetadata = {
    "input_tokens": int, # Total number of input or prompt tokens.
    "output_tokens": int, # Total number of generated output tokens.
    "total_tokens": int, # Total number of input and output tokens.
    "input_token_details": InputTokenDetails,  # Optional,, breakdown of input tokens, such as audio, cache-creation, or cache-read tokens.
    "output_token_details": OutputTokenDetails, # Optional breakdown of output tokens, such as audio or reasoning tokens.
}
```
The detailed token counts do not have to add up to the main token total, and providers may omit unsupported detail fields.

In [16]:
from langchain_core.messages import AIMessage, InvalidToolCall, ToolCall, UsageMetadata # Imports the message and supporting types.

valid_call = ToolCall( # Creates a successfully parsed tool request.
    type="tool_call", # Identifies it as a valid tool call.
    id="call_1", # Links the request with its future ToolMessage result.
    name="add", # Specifies the tool to execute.
    args={"a": 10, "b": 20}, # Stores the parsed tool arguments.
)

invalid_call = InvalidToolCall( # Stores a tool request that could not be parsed.
    type="invalid_tool_call", # Identifies it as an invalid tool call.
    id="call_2", # Stores the attempted tool-call identifier.
    name="multiply", # Stores the requested tool name.
    args='{"a": 10, "b": }', # Stores the malformed argument string.
    error="Invalid JSON arguments.", # Stores the parsing error.
)

usage = UsageMetadata( # Stores standardized token-usage information.
    input_tokens=12, # Stores the number of input tokens.
    output_tokens=8, # Stores the number of output tokens.
    total_tokens=20, # Stores the total token count.
)


print(valid_call) # Output: add
print(invalid_call) # Output: Invalid JSON arguments.
print(usage) # Output: 20

{'type': 'tool_call', 'id': 'call_1', 'name': 'add', 'args': {'a': 10, 'b': 20}}
{'type': 'invalid_tool_call', 'id': 'call_2', 'name': 'multiply', 'args': '{"a": 10, "b": }', 'error': 'Invalid JSON arguments.'}
{'input_tokens': 12, 'output_tokens': 8, 'total_tokens': 20}


# Methods

## `pretty_repr`
Returns a formatted string representation of the message.
* **Syntax:**
    ```python
    message.pretty_repr(
        html: bool = False,
        # False returns a plain-text representation.
        # True returns an HTML-formatted representation.
    ) -> str # Returns the formatted message without printing it.
    ```

In [10]:
print(message.pretty_repr())

================================== Ai Message ==================================

I will add the two numbers.
Tool Calls:
  add (call_1)
 Call ID: call_1
  Args:
    a: 10
    b: 20


## `pretty_print`
Prints a formatted representation of the message.
* **Syntax:**
    ```python
    message.pretty_print() -> None # Prints the message using formatting suitable for the current environment.
    ```

In [11]:
message.pretty_print()

================================== Ai Message ==================================

I will add the two numbers.
Tool Calls:
  add (call_1)
 Call ID: call_1
  Args:
    a: 10
    b: 20


## Message addition
Combines the message with another prompt-compatible value.
* **Syntax:**
```python
message + other -> ChatPromptTemplate
# Returns a ChatPromptTemplate containing this AIMessage
# followed by the supplied message or prompt component.
#
# It does not merge two AIMessages into one AIMessage.
```

In [12]:
from langchain_core.messages import AIMessage, HumanMessage # Imports AI and human message classes.

ai_message = AIMessage(content="The answer is 42.") # Creates the first AI message.
human_message = HumanMessage(content="Can you explain it?") # Creates the next human message.

prompt = ai_message + human_message # Combines both messages into a ChatPromptTemplate.
prompt_value = prompt.invoke({}) # Converts the template into a prompt value.

print(type(prompt).__name__) # Output: ChatPromptTemplate
print(prompt_value.messages[0].content) # Output: The answer is 42.
print(prompt_value.messages[1].content) # Output: Can you explain it?

ChatPromptTemplate
The answer is 42.
Can you explain it?


# Errors
`AIMessage` may raise:
1. `pydantic.ValidationError`: A constructor field has an incompatible type or invalid structure.
2. `NotImplementedError`: An unsupported object is combined with the message through the `+` operator.

In [14]:
from pydantic import ValidationError # Imports Pydantic's validation exception.
from langchain_core.messages import AIMessage # Imports the AIMessage class.

try:
    message = AIMessage(content={"invalid": "content"}) # Passes an unsupported content structure.
except ValidationError as error:
    print("ValidationError:", error.errors()[0]["msg"]) # Prints the validation error.

message = AIMessage(content="Hello") # Creates a valid AI message.

try:
    result = message + 123 # Tries to add an unsupported object.
except NotImplementedError as error:
    print("NotImplementedError:", error) # Prints the unsupported-addition error.

ValidationError: Input should be a valid string
NotImplementedError: Unsupported operand type for +: <class 'int'>
